# B2 — Transfer by heuristic: sentiment model + rule "negative ⇒ sarcastic"

`CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment`, no training. Two variants per preprocessing:
`hard` (argmax = negative, fixed 0.5) and `soft` (P(negative), threshold tuned on val).

**Contamination warning**: this model was fine-tuned on ASTD / ArSAS / SemEval, and ArSarcasm v1 (inside ArSarcasm-v2) re-annotated
ASTD and SemEval-2017 — it may already have seen some of our tweets.

In [ ]:
# Colab only: packages missing from the default image
%pip install -q emoji tashaphyne arabert pyarabic sentencepiece protobuf

In [ ]:
import os, sys

PROJECT = "/content/drive/MyDrive/tp2-arabic-sarcasm"   # <- where the repo lives in your Drive
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(PROJECT + "/notebooks")
sys.path.append("..")

In [ ]:
import numpy as np
import pandas as pd
import time
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.data import get_data
from src.evaluate import log_result, metrics, plot_results, save_predictions, show
from src.preprocess import build_pipeline
from src.train import best_threshold, set_seed

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

In [ ]:
train, val, test, report = get_data()
y_val = val["sarcasm"].values
print(f"val: {len(val)} | sarcasm rate: {y_val.mean():.3f}")

In [ ]:
BASE = ["html", "mentions", "hashtags"]
EMOJI = ["emoji"]
NORM = ["diacritics", "kashida", "letters", "elongation"]
LIGHT = ["light_stem"]
URLS = ["urls"]

COMBOS = {
    # same names / definitions as 02_baseline (so rows are comparable in results.csv)
    "base":                      BASE,
    "emoji":                     BASE + EMOJI,
    "norm":                      BASE + NORM,
    "light_stem":                BASE + LIGHT,
    "emoji + norm":              BASE + EMOJI + NORM,
    "emoji + light_stem":        BASE + EMOJI + LIGHT,
    "norm + light_stem":         BASE + NORM + LIGHT,
    "emoji + norm + light_stem": BASE + EMOJI + NORM + LIGHT,
    "urls kept":                 BASE,
    # added for PLMs: URLs -> <URL>, no stemming, AraBERT's own preprocessor
    "urls replaced":             BASE + URLS,
    "emoji + urls replaced":     BASE + URLS + EMOJI,
    "minimal":                   ["html", "urls", "mentions"],   # what 05_finetune uses
    "arabert":                   None,
}

In [ ]:
def make_prep(steps):
    if steps is None:
        from arabert.preprocess import ArabertPreprocessor
        return ArabertPreprocessor(model_name="aubmindlab/bert-base-arabertv02-twitter").preprocess
    return build_pipeline(steps)


def prepare(frame, steps):
    prep = make_prep(steps)
    texts = [prep(tweet) for tweet in frame["tweet"]]
    return [t if t.strip() else raw for t, raw in zip(texts, frame["tweet"])]


try:
    make_prep(None)
except Exception as error:
    print("arabert preprocessing skipped:", error)
    COMBOS.pop("arabert")

print(list(COMBOS))
print("base       :", prepare(val.head(4), COMBOS["base"])[3][:100])
print("urls kept == base (BASE has no urls step), so those two rows are expected to match")

In [ ]:
def slug(name):
    return name.replace(" + ", "+").replace(" ", "_")

NAME = "CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
tok = AutoTokenizer.from_pretrained(NAME)
clf = AutoModelForSequenceClassification.from_pretrained(NAME).to(DEVICE).eval()

print(clf.config.id2label)
NEG = [i for i, label in clf.config.id2label.items() if "neg" in label.lower()][0]
print("negative class id:", NEG)

In [ ]:
@torch.no_grad()
def sentiment_proba(texts, batch=64, max_len=128):
    order = np.argsort([len(t) for t in texts])
    proba = np.zeros((len(texts), clf.config.num_labels), dtype=np.float32)
    for i in range(0, len(texts), batch):
        idx = order[i:i + batch]
        enc = tok([texts[j] for j in idx], padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
        proba[idx] = torch.softmax(clf(**enc).logits.float(), dim=-1).cpu().numpy()
    return proba

In [ ]:
def log(model, name, proba, threshold, seconds):
    scores = metrics(y_val, proba, threshold)
    log_result(
        model=model,
        preprocessing=name,
        seed=42,
        threshold=threshold,
        train_time=0.0,
        trained_params=0,
        **{f"val_{k}": v for k, v in scores.items()}
    )
    save_predictions(f"{model}_{slug(name)}", 42, val["id"], y_val, proba, threshold)
    show(f"{model[-4:]} | {name}", scores, threshold, seconds)
    return {"model": model, "preprocessing": name, "threshold": threshold, **scores}


def run(combos):
    rows, seen = [], {}
    for name, steps in combos.items():
        texts = prepare(val, steps)
        h = hash(tuple(texts))

        start = time.time()
        if h not in seen:
            seen[h] = sentiment_proba(texts)
        proba = seen[h]
        seconds = time.time() - start

        hard = (proba.argmax(1) == NEG).astype(float)
        rows.append(log("camelbert_sent_hard", name, hard, 0.5, seconds))
        rows.append(log("camelbert_sent_soft", name, proba[:, NEG], best_threshold(y_val, proba[:, NEG]), seconds))

    return pd.DataFrame(rows)

table = run(COMBOS)

In [ ]:
table.pivot(index="preprocessing", columns="model", values="f1").round(4)

Ceiling of the heuristic: what if the sentiment were perfect? (gold `sentiment == NEG` ⇒ sarcastic)

In [ ]:
oracle = (val["sentiment"].str.upper() == "NEG").astype(float).values
print({k: round(v, 4) for k, v in metrics(y_val, oracle, 0.5).items()})
pd.crosstab(val["sentiment"], val["sarcasm"], normalize="index").round(3)

In [ ]:
summary = table.assign(name=table["model"] + " | " + table["preprocessing"]).set_index("name")
plot_results(summary.nlargest(15, "f1"))